# Notebook 01: Data Collection

## Purpose
Collect and save resolved prediction market metadata from Polymarket.
Markets are filtered to a date window (June 1 2025 — March 1 2026) that contains
all documented cases of insider trading used as validation targets in NB02.
Sports markets are excluded. Only order-book-enabled markets are kept.

No analysis is performed here. All feature engineering and clustering happens in NB02.

## K-Means Features (engineered in NB02)

- **Uncertainty at open** — how close to the fair even-split price the market opened
    - `opening_price` — first available price from CLOB API
    - `event_id` — used to count `n_outcomes` (number of markets in the same event)

- **Time to resolution** — total duration the market was open in hours
    - `duration_hrs` — derived from first and last trade timestamps via Dune

- **Trading intensity** — average dollars traded per hour
    - `total_volume` — total dollars traded (from Dune)
    - `duration_hrs` — market duration in hours (from Dune)

- **Average trade size** — mean dollars per trade
    - `avg_trade_size` — from Dune

- **Late capital concentration** — proportion of volume in final 48 hours
    - `late_capital_ratio` — from Dune

- **Trade concentration** — fraction of volume from top 3 wallets
    - `top3_wallet_concentration` — from Dune

## Additional fields collected
- `market_id` — unique conditionId
- `question` — market description
- `category` — topic tag
- `end_date` — scheduled close date
- `closed_time` — actual close date
- `token_ids` — CLOB token IDs, used to fetch opening price

## Outputs
- `data/raw/markets_polymarket.parquet` — one row per market with all features

## APIs
- Polymarket Gamma API: gamma-api.polymarket.com (market metadata)
- Polymarket CLOB API: clob.polymarket.com (opening price)
- Dune Analytics: dune.com (trade-level features via polymarket_polygon.market_trades)

In [49]:
# Cell 2 — Imports and setup
import requests
import pandas as pd
import time
from pathlib import Path
from collections import Counter
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import sys
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("DUNE_API_KEY")[:5])  # prints first 5 chars to confirm it loaded


Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

SPORTS_CATEGORIES = {"Sports", "Tennis", "Formula 1", "MLB", "Golf", "Racing", "Cricket", "Boxing", "MMA"}
SPORTS_TAG_IDS = {1, 232, 1005, 1006, 1012}
SPORTS_KEYWORDS = ["champions league", "uefa", "nfl", "nba", "mlb", "super bowl", "ufc", "fifa"]

print("Ready.")

h4B1Q
Ready.


In [50]:
# Cell 3: Get Polymarket Markets

def fetch_all_polymarket_events(limit=100):
    """
    Fetches resolved Polymarket events globally, ordered by volume.
    """
    base_url = "https://gamma-api.polymarket.com/events"
    all_events = []
    offset = 0
    page = 0

    while True:
        params = {
            "closed": "true",
            "order": "volume",
            "ascending": "false",
            "limit": limit,
            "offset": offset
        }
        
        try:
            resp = requests.get(base_url, params=params)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"Stopped at page {page}, offset {offset}: {e}")
            break
            
        if not data:
            break
            
        for event in data:
            event_tags = {tag.get("id") for tag in event.get("tags", [])}
            title_lower = event.get("title", "").lower()
            
            if event_tags.intersection(SPORTS_TAG_IDS):
                continue
            if any(keyword in title_lower for keyword in SPORTS_KEYWORDS):
                continue
                
            all_events.append(event)
            
        if len(data) < limit:
            break
            
        offset += limit
        page += 1
        time.sleep(0.1)

    return all_events

# Execute global fetch
raw_events = fetch_all_polymarket_events()

# Deduplicate by event id
seen = set()
unique_events = []
for e in raw_events:
    if e["id"] not in seen:
        seen.add(e["id"])
        unique_events.append(e)

print(f"\nTotal unique non-sports events collected: {len(unique_events)}")

# Count the occurrences of each category/tag
tag_counts = Counter()
tag_id_to_label = {}

for event in unique_events:
    for tag in event.get("tags", []):
        tag_label = tag.get("label")
        tag_id = tag.get("id")
        if tag_label:
            tag_counts[tag_label] += 1
            tag_id_to_label[tag_label] = tag_id

print("\n--- Event Breakdown by Category ---")
if not tag_counts:
    print("No tags found. Check if the 'tags' key exists in the API response payload.")
else:
    for tag_label, count in tag_counts.most_common():
        print(f"• {tag_label} (ID: {tag_id_to_label[tag_label]}): {count} events")

Stopped at page 21, offset 2100: 422 Client Error: Unprocessable Entity for url: https://gamma-api.polymarket.com/events?closed=true&order=volume&ascending=false&limit=100&offset=2100

Total unique non-sports events collected: 2054

--- Event Breakdown by Category ---
• Sports (ID: 1): 1073 events
• Games (ID: 100639): 999 events
• Politics (ID: 2): 515 events
• Basketball (ID: 28): 393 events
• NBA (ID: 745): 382 events
• Soccer (ID: 100350): 377 events
• Crypto (ID: 21): 277 events
• FIFA World Cup (ID: 102232): 242 events
• Crypto Prices (ID: 1312): 227 events
• Recurring (ID: 101757): 195 events
• Geopolitics (ID: 100265): 180 events
• Bitcoin (ID: 235): 177 events
• Culture (ID: 596): 164 events
• World (ID: 101970): 154 events
• Trump (ID: 126): 142 events
• Hide From New (ID: 102169): 135 events
• Weekly (ID: 102264): 117 events
• Tweet Markets (ID: 972): 115 events
• NFL (ID: 450): 111 events
• Multi Strikes (ID: 102516): 104 events
• Esports (ID: 64): 103 events
• Iran (ID: 78

In [51]:
# Cell 4 — Extract markets from events, filter by date/category, and save metadata

def parse_polymarket_market(market, event):
    token_ids = market.get("clobTokenIds", "[]")
    if isinstance(token_ids, str):
        try:
            token_ids = json.loads(token_ids)
        except json.JSONDecodeError:
            token_ids = []
            
    # Safely extract the primary tag label as the category
    tags = event.get("tags", [])
    category = tags[0].get("label") if isinstance(tags, list) and len(tags) > 0 else None

    return {
        "market_id": market.get("conditionId"),
        "event_id": event.get("id"),
        "question": market.get("question"),
        "category": category,
        "start_date": event.get("startDate"),
        "end_date": event.get("endDate"),
        "closed_time": event.get("closedTime"),
        "volume": market.get("volume"),
        "liquidity": market.get("liquidity"),
        "enable_order_book": market.get("enableOrderBook"),
        "token_ids": token_ids,
        "platform": "polymarket"
    }

pm_markets = []
for event in unique_events:
    for market in event.get("markets", []):
        pm_markets.append(parse_polymarket_market(market, event))

# Convert to DataFrame
pm_markets_df = pd.DataFrame(pm_markets)

# 1. Clean data and enforce types
pm_markets_df["volume"] = pd.to_numeric(pm_markets_df["volume"], errors="coerce")
pm_markets_df["liquidity"] = pd.to_numeric(pm_markets_df["liquidity"], errors="coerce")
pm_markets_df["closed_time"] = pd.to_datetime(pm_markets_df["closed_time"], errors="coerce", utc=True)
pm_markets_df["start_date"] = pd.to_datetime(pm_markets_df["start_date"], errors="coerce", utc=True)
pm_markets_df["end_date"] = pd.to_datetime(pm_markets_df["end_date"], errors="coerce", utc=True)

# 2. Strict Category and Keyword Filtering (Purge Sports)
# Filter by the category column text
pm_markets_df = pm_markets_df[~pm_markets_df["category"].isin(SPORTS_CATEGORIES)]

# Extra safety: Filter out common sports terms from the question column
SPORTS_WORDS = ["champions league", "uefa", "nfl", "nba", "mlb", "super bowl", "ufc", "fifa"]
sports_regex = "|".join(SPORTS_WORDS)
pm_markets_df = pm_markets_df[~pm_markets_df["question"].str.lower().str.contains(sports_regex, na=False)]

# 3. Keep only markets with order book enabled
pm_markets_df = pm_markets_df[pm_markets_df["enable_order_book"] == True].copy()

# 4. Apply the 6/1/2025 to 2/25/2026 time-window filter
start_window = pd.to_datetime("2025-06-01", utc=True)
end_window = pd.to_datetime("2026-03-01 23:59:59", utc=True)

date_mask = (pm_markets_df["closed_time"] >= start_window) & (pm_markets_df["closed_time"] <= end_window)
filtered_pm_markets_df = pm_markets_df[date_mask].copy()

# Derive n_outcomes from event_id grouping
filtered_pm_markets_df["n_outcomes"] = filtered_pm_markets_df.groupby("event_id")["market_id"].transform("count")

# Save final clean dataset
filtered_pm_markets_df.to_parquet("../data/raw/markets_polymarket.parquet", index=False)

print(f"Total non-sports order-book markets: {len(pm_markets_df)}")
print(f"Markets matching test window (6/1/25 - 3/1/26): {len(filtered_pm_markets_df)}")
filtered_pm_markets_df.head()


Total non-sports order-book markets: 18969
Markets matching test window (6/1/25 - 3/1/26): 7623


,market_id,event_id,question,category,start_date,end_date,closed_time,volume,liquidity,enable_order_book,token_ids,platform,n_outcomes
70,0x17815081230e3b9c78b098162c33b1ffa68c4ec29c12...,45883,Fed decreases interest rates by 50+ bps after ...,fomc,NaT,2026-01-28 00:00:00+00:00,2026-01-28 23:04:04+00:00,2.350652e+08,NaN,True,[118621655667573459852404761644897182190567350...,polymarket,4
71,0x35cc41270f5cdfd59b45e68ab85dc51b0b900d286a6c...,45883,Fed decreases interest rates by 25 bps after J...,fomc,NaT,2026-01-28 00:00:00+00:00,2026-01-28 23:04:04+00:00,1.012115e+08,NaN,True,[927037616823224806649767662476141278780239886...,polymarket,4
72,0xe93c89c41d1bb08d3bb40066d8565df301a696563b25...,45883,No change in Fed interest rates after January ...,fomc,NaT,2026-01-28 00:00:00+00:00,2026-01-28 23:04:04+00:00,1.067671e+08,NaN,True,[112838095111461683880944516726938163688341306...,polymarket,4
73,0x7c6c69d91b21cbbea08a13d0ad51c0e96a956045aaad...,45883,Fed increases interest rates by 25+ bps after ...,fomc,NaT,2026-01-28 00:00:00+00:00,2026-01-28 23:04:04+00:00,2.164557e+08,NaN,True,[164196493540672984127369198307778307300266774...,polymarket,4
130,0xc40e1cfe1b34a6062a93583328707d6d363d63b5d023...,114242,"US strikes Iran by December 31, 2025?",Parent For Derivative,NaT,2026-06-30 00:00:00+00:00,2026-02-28 10:03:13+00:00,1.919312e+04,NaN,True,[443724883773351594016161705918439451498161380...,polymarket,65


In [52]:
# Cell 5: Getting the starting price for each market using a Multithreaded Session
# Initialize a highly optimized, thread-safe connection pool session
session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=50, pool_maxsize=50)
session.mount('https://', adapter)

def get_opening_price(row):
    """
    Safely isolates the first string item from the token list to prevent bracket errors,
    and pulls historical lifecycles asynchronously using the working GET logic.
    """
    try:
        # Extract token list from row
        token_ids = row.get('token_ids', [])
        if not isinstance(token_ids, list) or len(token_ids) == 0:
            return row.name, None
        
        # FIX: Isolate the first element out of the list as a clean string string
        # This removes brackets ['...'] which broke the batch endpoint
        yes_token_id = str(token_ids[0]).strip()
        
        # API request setup using the exact parameters verified by your test cell
        url = "https://clob.polymarket.com/prices-history"
        params = {
            "market": yes_token_id,
            "interval": "max",
            "fidelity": 1440  # Daily bars to maximize safety and speed
        }
        
        # Execute query utilizing pooled network sessions
        response = session.get(url, params=params, timeout=12)
        
        if response.status_code == 200:
            raw_data = response.json()
            
            # Extract history list depending on cached structure payload variant
            history_list = raw_data.get('history', []) if isinstance(raw_data, dict) else raw_data
            
            if isinstance(history_list, list) and len(history_list) > 0:
                # First element in max interval is chronologically the oldest opening candle
                first_point = history_list[0]
                opening_price = float(first_point.get('p'))
                return row.name, opening_price
                
        return row.name, None
        
    except Exception as e:
        # Secure background thread debugger stream
        sys.stdout.write(f"[Thread Alert] Index {row.name} failed locally: {str(e)}\n")
        sys.stdout.flush()
        return row.name, None

# Dictionary to hold mappings
results = {}

# Set to 15 concurrent workers to rapidly cycle network I/O bounds safely
max_workers = 15
rows = [row for _, row in filtered_pm_markets_df.iterrows()]

print(f"Starting parallel API extraction for {len(rows)} markets using {max_workers} threads...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(get_opening_price, row): row.name for row in rows}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching Opening Prices"):
        idx, price = future.result()
        results[idx] = price

# Close down pooled connection handles safely
session.close()

# Map calculated pricing series straight back into original Dataframe tracking rows
filtered_pm_markets_df['opening_price'] = filtered_pm_markets_df.index.map(results)

# Define output target path
output_path = "../data/raw/markets_polymarket.parquet"

# Save updated DataFrame configuration to a compressed Parquet layout
filtered_pm_markets_df.to_parquet(output_path, index=False)

# Compile performance statistics metrics
total_markets = len(filtered_pm_markets_df)
successful_fetches = filtered_pm_markets_df['opening_price'].notna().sum()
failed_fetches = filtered_pm_markets_df['opening_price'].isna().sum()

print("\n" + "=" * 40)
print("FETCHING SUMMARY COMPLETE")
print("=" * 40)
print(f"Total Markets Processed: {total_markets:,}")
print(f"Successfully Fetched:    {successful_fetches:,} ({successful_fetches / total_markets:.2%})")
print(f"Failed / Missing Prices: {failed_fetches:,} ({failed_fetches / total_markets:.2%})")
print(f"File Saved Successfully to: {output_path}")
print("=" * 40)



Starting parallel API extraction for 7623 markets using 15 threads...


Fetching Opening Prices:   0%|          | 0/7623 [00:00<?, ?it/s]


FETCHING SUMMARY COMPLETE
Total Markets Processed: 7,623
Successfully Fetched:    6,184 (81.12%)
Failed / Missing Prices: 1,439 (18.88%)
File Saved Successfully to: ../data/raw/markets_polymarket.parquet


In [53]:
# Cell 6 — Comprehensive Pipeline Sanity Check
import pandas as pd
import numpy as np

# 1. Load the saved Parquet file
data_path = "../data/raw/markets_polymarket.parquet"
try:
    check_df = pd.read_parquet(data_path)
    print(f"✅ File loaded successfully. Total records to analyze: {len(check_df)}\n")
except Exception as e:
    print(f"❌ Critical Failure: Could not load Parquet file. Error: {e}")
    check_df = pd.DataFrame()

if not check_df.empty:
    print("=" * 50)
    print("        DATA INTEGRITY METRICS & AUDIT        ")
    print("=" * 50)
    
    # Check A: Missing Data and Price Pull Success Rates
    total = len(check_df)
    missing_prices = check_df['opening_price'].isna().sum()
    missing_tokens = check_df['token_ids'].apply(lambda x: len(x) == 0 if isinstance(x, (list, np.ndarray)) else True).sum()
    
    print(f"📊 Extraction Coverage:")
    print(f"• Opening Prices Successfully Captured: {total - missing_prices}/{total} ({(total - missing_prices)/total:.1%})")
    print(f"• Markets Missing CLOB Token IDs:      {missing_tokens}/{total} ({missing_tokens/total:.1%})")
    
    # Check B: Strict Date Window Compliance Check
    # (Checking if data strictly matches your June 1, 2025 to March 1, 2026 window)
    start_bound = pd.to_datetime("2025-06-01", utc=True)
    end_bound = pd.to_datetime("2026-03-01 23:59:59", utc=True)
    
    out_of_bounds_early = (check_df['closed_time'] < start_bound).sum()
    out_of_bounds_late = (check_df['closed_time'] > end_bound).sum()
    
    print(f"\n⏳ Chronological Bounding Verification:")
    print(f"• Earliest Market 'closed_time' found:  {check_df['closed_time'].min()}")
    print(f"• Latest Market 'closed_time' found:    {check_df['closed_time'].max()}")
    if out_of_bounds_early > 0 or out_of_bounds_late > 0:
        print(f"⚠️  Warning: {out_of_bounds_early} markets closed before June 2025. {out_of_bounds_late} closed after March 1, 2026.")
    else:
        print(f"• Strict Time Frame Compliance:        Passed (All rows inside target window)")

    # Check C: Sports Leakage Audit
    # Scan remaining questions for hidden sports terms that escaped Cell 3 or Cell 4
    audit_sports_words = ["vs", "match", "game", "team", "points", "win", "playoffs", "score"]
    sports_regex = "|".join(audit_sports_words)
    suspected_sports = check_df['question'].str.lower().str.contains(sports_regex, na=False).sum()
    
    print(f"\n🛡️  Anti-Sports Leakage Audit:")
    print(f"• Markets explicitly labeled 'Sports':  {len(check_df[check_df['category'] == 'Sports'])}")
    print(f"• Markets with suspected sports terms: {suspected_sports}/{total}")
    if suspected_sports > 0:
        print("💡 Review sample flagged questions below to verify they are non-sports (e.g. politics or tech 'games').")

    print(f"\n")
    
    df = pd.read_parquet("../data/raw/markets_polymarket.parquet")

    for kw in ["maduro", "iran", "state of the union", "d4vd", "santos"]:
        n = df["question"].str.contains(kw, case=False, na=False).sum()
        print(f"{kw}: {n} markets")

    # Check d4vd by conditionId
    d4vd_id = "0xeaf59fcbf65e45abac0383dad483239d849e6d48d9eb2a6b3bf5cc1c7e9cf2ad"
    print(f"\nd4vd by conditionId: {df['market_id'].eq(d4vd_id).sum()}")

    # Check D: Sample Review Panel
    print("\n" + "=" * 50)
    print("          DATAFRAME STRUCTURAL SNAPSHOT          ")
    print("=" * 50)
    display_cols = ['market_id', 'question', 'category', 'closed_time', 'opening_price']
    # Safely print whatever valid columns exist out of the requested checklist
    existing_cols = [c for c in display_cols if c in check_df.columns]
    display(check_df[existing_cols].head(3))


✅ File loaded successfully. Total records to analyze: 7623

        DATA INTEGRITY METRICS & AUDIT        
📊 Extraction Coverage:
• Opening Prices Successfully Captured: 6184/7623 (81.1%)
• Markets Missing CLOB Token IDs:      25/7623 (0.3%)

⏳ Chronological Bounding Verification:
• Earliest Market 'closed_time' found:  2025-06-01 07:20:27+00:00
• Latest Market 'closed_time' found:    2026-03-01 20:46:17+00:00
• Strict Time Frame Compliance:        Passed (All rows inside target window)

🛡️  Anti-Sports Leakage Audit:
• Markets explicitly labeled 'Sports':  0
• Markets with suspected sports terms: 1959/7623
💡 Review sample flagged questions below to verify they are non-sports (e.g. politics or tech 'games').


maduro: 12 markets
iran: 213 markets
state of the union: 51 markets
d4vd: 2 markets
santos: 2 markets

d4vd by conditionId: 1

          DATAFRAME STRUCTURAL SNAPSHOT          


,market_id,question,category,closed_time,opening_price
0,0x17815081230e3b9c78b098162c33b1ffa68c4ec29c12...,Fed decreases interest rates by 50+ bps after ...,fomc,2026-01-28 23:04:04+00:00,0.120
1,0x35cc41270f5cdfd59b45e68ab85dc51b0b900d286a6c...,Fed decreases interest rates by 25 bps after J...,fomc,2026-01-28 23:04:04+00:00,0.545
2,0xe93c89c41d1bb08d3bb40066d8565df301a696563b25...,No change in Fed interest rates after January ...,fomc,2026-01-28 23:04:04+00:00,0.495


In [54]:
df = pd.read_parquet("../data/raw/markets_polymarket.parquet")
df[["market_id"]].rename(columns={"market_id": "condition_id"}).to_csv("../data/raw/condition_ids.csv", index=False)
print(f"Saved {len(df)} condition IDs")

Saved 7623 condition IDs


In [63]:
from dune_client.client import DuneClient
import pandas as pd

# DUNE_API_KEY = environment variable
# QUERY_ID = evironment variable

dune = DuneClient(DUNE_API_KEY)
result = dune.get_latest_result(QUERY_ID, max_age_hours=1)
dune_df = pd.DataFrame(result.result.rows)

# Drop columns from dune_df that already exist in parquet
dune_df = dune_df.drop(columns=["question", "condition_id"], errors="ignore")
dune_df = dune_df.rename(columns={"condition_id": "market_id"}) if "condition_id" in dune_df.columns else dune_df

# Load parquet and strip all previous Dune columns and merge artifacts
pm_df = pd.read_parquet("../data/raw/markets_polymarket.parquet")
cols_to_drop = ["avg_trade_size", "duration_hrs", "first_trade", "last_trade",
                "late_capital_ratio", "top3_wallet_concentration", "total_volume", 
                "trade_count", "condition_id", "question_x", "question_y"]
pm_df = pm_df.drop(columns=[c for c in cols_to_drop if c in pm_df.columns])
pm_df = pm_df.rename(columns={"question_x": "question"}) if "question_x" in pm_df.columns else pm_df

# Re-fetch dune_df with condition_id for merge key
result2 = dune.get_latest_result(QUERY_ID)
dune_df2 = pd.DataFrame(result2.result.rows)
dune_df2 = dune_df2.drop(columns=["question"], errors="ignore")

merged = pm_df.merge(dune_df2, left_on="market_id", right_on="condition_id", how="left")
merged = merged.drop(columns=["condition_id"], errors="ignore")
merged.to_parquet("../data/raw/markets_polymarket.parquet", index=False)

print(f"Total markets: {len(merged)}")
print(f"Markets with Dune data: {merged['total_volume'].notna().sum()}")
print(merged.columns.tolist())

Total markets: 7623
Markets with Dune data: 6592
['market_id', 'event_id', 'question', 'category', 'start_date', 'end_date', 'closed_time', 'volume', 'liquidity', 'enable_order_book', 'token_ids', 'platform', 'n_outcomes', 'opening_price', 'avg_trade_size', 'duration_hrs', 'first_trade', 'last_trade', 'late_capital_ratio', 'top3_wallet_concentration', 'total_volume', 'trade_count']


In [66]:
df = pd.read_parquet("../data/raw/markets_polymarket.parquet")
print(f"Total markets: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values:")
print(df[["opening_price", "n_outcomes", "total_volume", "duration_hrs", "late_capital_ratio", "top3_wallet_concentration", "avg_trade_size"]].isna().sum())

d4vd = df[df["market_id"] == "0xeaf59fcbf65e45abac0383dad483239d849e6d48d9eb2a6b3bf5cc1c7e9cf2ad"]
print(f"\nd4vd features:")
print(d4vd[["opening_price", "n_outcomes", "total_volume", "duration_hrs", "late_capital_ratio", "top3_wallet_concentration", "avg_trade_size"]].to_string())

Total markets: 7623
Columns: ['market_id', 'event_id', 'question', 'category', 'start_date', 'end_date', 'closed_time', 'volume', 'liquidity', 'enable_order_book', 'token_ids', 'platform', 'n_outcomes', 'opening_price', 'avg_trade_size', 'duration_hrs', 'first_trade', 'last_trade', 'late_capital_ratio', 'top3_wallet_concentration', 'total_volume', 'trade_count']

Missing values:
opening_price                1439
n_outcomes                      0
total_volume                 1031
duration_hrs                 1031
late_capital_ratio           1031
top3_wallet_concentration    1031
avg_trade_size               1031
dtype: int64

d4vd features:
      opening_price  n_outcomes  total_volume  duration_hrs  late_capital_ratio  top3_wallet_concentration  avg_trade_size
3417          0.095          47  173834.84367         337.0            0.785877                   0.737554      163.840569


In [58]:
df = pd.read_parquet("../data/raw/markets_polymarket.parquet")
print(df.columns.tolist())
print(f"Total rows: {len(df)}")

['market_id', 'event_id', 'question', 'category', 'start_date', 'end_date', 'closed_time', 'volume', 'liquidity', 'enable_order_book', 'token_ids', 'platform', 'n_outcomes', 'opening_price']
Total rows: 7623


In [67]:
import shutil
shutil.copy(
    "../data/raw/markets_polymarket.parquet",
    "../data/raw/markets_polymarket_backup.parquet"
)

'../data/raw/markets_polymarket_backup.parquet'

In [68]:
from dune_client.client import DuneClient
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# DUNE_API_KEY = environment variable
# VELOCITY_QUERY_ID = environment variable

dune = DuneClient(DUNE_API_KEY)
result = dune.get_latest_result(VELOCITY_QUERY_ID)
velocity_df = pd.DataFrame(result.result.rows)

print(f"Velocity rows returned: {len(velocity_df)}")
print(velocity_df.head(3))

pm_df = pd.read_parquet("../data/raw/markets_polymarket.parquet")
pm_df = pm_df.drop(columns=["early_avg_change", "late_avg_change", "velocity_ratio"],
                    errors="ignore")

merged = pm_df.merge(
    velocity_df[["condition_id", "early_avg_change", "late_avg_change", "velocity_ratio"]],
    left_on="market_id",
    right_on="condition_id",
    how="left"
).drop(columns=["condition_id"], errors="ignore")

merged.to_parquet("../data/raw/markets_polymarket.parquet", index=False)

print(f"Total markets: {len(merged)}")
print(f"Markets with velocity ratio: {merged['velocity_ratio'].notna().sum()}")

d4vd = merged[merged["market_id"] == "0xeaf59fcbf65e45abac0383dad483239d849e6d48d9eb2a6b3bf5cc1c7e9cf2ad"]
print(f"\nd4vd velocity ratio: {d4vd['velocity_ratio'].values[0]}")

Velocity rows returned: 6751
                                        condition_id  early_avg_change  \
0  0x0e4f635c4fdb0fbfe986ddc93712388065abad3bf0f1...          0.737300   
1  0x17f753b4f078cf6dc39332ab6d3ae031d99c717227b0...          0.715833   
2  0x3f98a15cf10d4297c36571412d57c82ffc8ebf5d6ba0...          0.717300   

   late_avg_change  velocity_ratio  
0         0.753303        1.021705  
1         0.743107        1.038102  
2         0.751927        1.048273  
Total markets: 7623
Markets with velocity ratio: 6749

d4vd velocity ratio: 1.0781880143805245


In [ ]:
# making sure to collect negRisk

import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

df = pd.read_parquet("../data/raw/markets_polymarket.parquet")

def fetch_neg_risk(row):
    try:
        resp = requests.get(
            "https://gamma-api.polymarket.com/events",
            params={"id": row["event_id"]},
            timeout=10
        )
        data = resp.json()
        if data:
            for market in data[0].get("markets", []):
                if market.get("conditionId") == row["market_id"]:
                    return row.name, market.get("negRisk", False)
        return row.name, False
    except Exception:
        return row.name, False

results = {}
rows = [row for _, row in df.iterrows()]

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_neg_risk, row): row.name for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching negRisk"):
        idx, neg_risk = future.result()
        results[idx] = neg_risk

df["neg_risk"] = df.index.map(results)
df.to_parquet("../data/raw/markets_polymarket.parquet", index=False)

print(f"neg_risk coverage: {df['neg_risk'].notna().sum()} / {len(df)}")
print(df["neg_risk"].value_counts())